# 5.1 Course Bottleneck Detection with Unsupervised Learning

## Course 3: Advanced Classification Models for Student Success

## Opening Narrative

> *"We've spent this course predicting who will leave. Now we ask a different question:
> **Which courses are quietly working against our students?**"*

### The Provost's Question

Imagine your Provost says:

> *"Some courses seem to act as **bottlenecks** — students hit them, fail or withdraw,
> repeat them a semester or two later, and lose momentum toward their degree.
> We have data on 1,000 course sections. Can we **discover** which courses are the
> real bottlenecks — and rank them so the Curriculum Committee knows where to start?"*

**That question is the heart of unsupervised learning.**

Unlike supervised learning (Modules 1–4), where we predicted a known target (`DEPARTED`),
unsupervised learning has **no target variable**. No one has labeled which courses are
"bottlenecks." Instead, we let the data reveal its own structure — clusters of similar
course sections — and then translate those patterns into an action plan.

### What You Will Learn

| Concept | What It Does | Institutional Use |
|:--------|:-------------|:------------------|
| **K-Means Clustering** | Groups similar observations into k clusters | Segment course sections into distinct profiles |
| **PCA** (Principal Component Analysis) | Reduces many variables to a few key dimensions | Visualize 12-variable course data in 2D |
| **Silhouette Score** | Helps gauge how many clusters the data supports | Sanity-check the number of course profiles for leadership |
| **Cluster Profiling** | Describes what makes each cluster unique | Translate patterns into curriculum-reform priorities |
| **Risk Prioritization Index** | Combines metrics into one ranked score | Hand leadership a ranked list of courses to review |

### Key Difference from Supervised Learning

| | Supervised (Modules 1–4) | Unsupervised (This Module) |
|:--|:------------------------|:--------------------------|
| **Goal** | Predict a known outcome | Discover hidden structure |
| **Target variable** | Yes (`DEPARTED`) | No |
| **Evaluation** | AUC, F1, Accuracy | Silhouette + Interpretability |
| **Question** | "Will this student leave?" | "What *kinds* of courses do we have?" |
| **Action** | Flag individual students | Prioritize course-level reform |

## 1. Setup and Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Visualization defaults
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("All libraries loaded successfully!")
print("This module uses matplotlib/seaborn for static, publication-ready visuals.")

## 2. Load the Course Section Data

The dataset lives in Google Drive alongside the other Course 3 data. We mount Drive,
point at the `data/` folder, and read `bottleneck.csv`.

> **Note:** In earlier modules we *generated* synthetic data inline. Here the synthetic
> course-section data has been pre-built and saved to `bottleneck.csv` so every run of
> this notebook analyzes exactly the same 1,000 sections.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set up file paths
project_path = '/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3'
data_filepath = '/data/'

# Load the course section data
course_df = pd.read_csv(f'{project_path}{data_filepath}bottleneck.csv')

print(f"Dataset: {course_df.shape[0]:,} course sections × {course_df.shape[1]} variables")
course_df.head()

### Variables (12)

| Variable | Description |
|:---------|:------------|
| `ENROLLMENT` | Section enrollment |
| `DFW_RATE` | DFW rate for the section (D, F, or Withdraw) |
| `PASS_RATE` | Pass rate (C or better) |
| `AVG_GPA` | Average grade in the section |
| `REPEAT_RATE` | Proportion of students repeating |
| `AVG_REPEAT_DELAY` | Average semesters before repeat |
| `SECTION_SIZE` | Number of seats |
| `PCT_FIRST_YEAR` | Proportion of first-year students |
| `PCT_STEM` | Proportion of STEM majors |
| `PREREQUISITE_COUNT` | Number of prerequisites |
| `IS_GATEWAY` | Gateway course flag (0/1) |
| `UNITS` | Course unit value |

In [ ]:
# Quick summary of the raw variables
course_df.describe().round(2)

## 3. What Is K-Means Clustering? (The Intuition)

Before any code, here's the idea in plain terms.

Imagine you dumped all 1,000 course sections onto a giant gym floor, and courses that
"look alike" (similar DFW rates, pass rates, enrollment, etc.) end up standing near each
other. **K-Means clustering** is just an automated way to draw *k* circles around the
natural huddles of courses.

Here's the loop it runs, which is easy to picture:

1. **Drop `k` flags on the floor at random.** Each flag is a tentative "team captain"
   (the technical name is a *centroid*).
2. **Every course joins its nearest flag.** Now you have `k` rough teams.
3. **Each flag walks to the middle of its team.** The captain repositions to the average
   location of everyone who joined it.
4. **Repeat steps 2–3** — courses re-pick their nearest flag, flags re-center — until
   nobody switches teams anymore. The huddles have settled.

That's it. K-Means keeps nudging the flags until each one sits at the heart of a tight
group of similar courses. The **"k"** is *how many groups you ask for* — you choose it
(we'll land on 4). "**Means**" refers to step 3: each flag moves to the *mean* of its team.

**Two things that make the intuition honest:**

- **K-Means only understands distance**, so every variable must be on a comparable scale
  first — otherwise a big-numbered variable like `ENROLLMENT` (in the hundreds) would
  drown out `DFW_RATE` (between 0 and 1). That's why **Step 1 below is scaling.**
- **K-Means doesn't know what a "bottleneck" is.** It just finds groups of look-alike
  courses. *We* interpret which group is the bottleneck afterward, in the profiling step.

Now let's run the pipeline that turns that idea into an answer.

### The Clustering Pipeline

We follow the same five-step pipeline used throughout this module:

**Scale → Choose k → Fit K-Means → Visualize with PCA → Profile the clusters.**

### Step 1: Scale the Data

As noted above, K-Means measures distance, so variables on large scales (like
`ENROLLMENT`, in the hundreds) would dominate variables on small scales (like `DFW_RATE`,
between 0 and 1). `StandardScaler` puts every variable on a comparable footing
(mean 0, standard deviation 1) so each one gets an equal vote in "who's near whom."

In [ ]:
scaler = StandardScaler()
course_scaled = scaler.fit_transform(course_df)

print(f"Scaled matrix shape: {course_scaled.shape}")
print(f"Mean of each scaled column (~0): {course_scaled.mean(axis=0).round(2)[:5]} ...")
print(f"Std of each scaled column  (~1): {course_scaled.std(axis=0).round(2)[:5]} ...")

### Step 2: Choosing the Number of Clusters

We sweep `k` from 2 to 10 and compute the **silhouette score** at each value
(higher = better-separated clusters). But the silhouette score is only *one* input to
the decision. As you'll see below, the metric alone does **not** hand us a single answer
— and blindly following it would give us fewer, less useful segments.

We will land on **k = 4**, and the two cells after the plot explain exactly why that is a
defensible choice even though it is not the silhouette-maximizing one.

In [ ]:
# Silhouette analysis across candidate k values
sils_c = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    sils_c.append(silhouette_score(course_scaled, km.fit_predict(course_scaled)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(2, 11), sils_c, 'go-', linewidth=2, markersize=8)
ax.set_xlabel('k'); ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Analysis — Course Sections')
ax.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k = 4 (chosen)')
ax.legend(); plt.tight_layout(); plt.show()

#### Why k = 4 when the silhouette peaks at k = 3?

Read the plot honestly: the silhouette score is **highest at k = 3** (≈ 0.38) and
*drops* to ≈ 0.28 at k = 4. Taken at face value, the metric says "use 3 clusters." So
why do we override it and pick 4?

Because **the silhouette score answers a narrower question than we do.** It measures only
one thing — how geometrically compact and well-separated the clusters are. It knows
nothing about whether the clusters are *useful* to a Curriculum Committee. Three facts,
which we verify with data in the next cell, resolve the tension:

1. **The bottleneck cluster is identical at k = 3 and k = 4.** The high-risk group
   (~200 sections: DFW ≈ 0.52, pass ≈ 0.47, repeat ≈ 0.37) is the *same sections*
   either way. **Our answer to the Provost's question does not depend on k.** That group
   is stable and trustworthy.

2. **Going from 3 → 4 clusters splits the large "healthy" bucket into two real course
   types**, not noise: *small upper-division* courses (few first-years, many
   prerequisites, tiny sections) versus *healthy mid-size core* courses. A dean treats
   those two groups differently, so keeping them separate is operationally valuable.

3. **The silhouette dip is expected and mild.** When you split one blob into two genuine
   but *adjacent* sub-groups, silhouette always falls a little — the new clusters sit
   closer to each other than either sits to the bottleneck group. A modest drop
   (0.38 → 0.28) is the price of a more actionable segmentation, not evidence of a bad split.

**The rule to internalize:** silhouette *narrows the field* (here, to 3–4 clusters — both
clearly beat k ≥ 5). **Interpretability makes the final call.** Never let a single metric
overrule what the clusters actually mean for the decision you're supporting.

Let's confirm claims (1) and (2) directly.

In [ ]:
# Compare k=3 vs k=4 side by side to justify the choice.
# Key columns for reading the story: outcomes + who takes the course.
compare_cols = ['ENROLLMENT', 'DFW_RATE', 'PASS_RATE', 'REPEAT_RATE',
                'PCT_FIRST_YEAR', 'IS_GATEWAY', 'PREREQUISITE_COUNT']

for k in (3, 4):
    km_tmp = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels_tmp = km_tmp.fit_predict(course_scaled)
    prof_tmp = course_df[compare_cols].groupby(labels_tmp).mean().round(2)
    prof_tmp['n'] = pd.Series(labels_tmp).value_counts().sort_index().values
    print("=" * 90)
    print(f"k = {k}  |  silhouette = {silhouette_score(course_scaled, labels_tmp):.3f}")
    print("=" * 90)
    print(prof_tmp.to_string())
    print()

print("Read the tables:")
print("  • The high-DFW / low-PASS bottleneck row (~200 sections) is IDENTICAL in both.")
print("  • At k=4, the single large healthy group at k=3 splits into two: a small,")
print("    high-prerequisite upper-division group and a mid-size core group.")
print("  • That extra split is interpretable and useful — so we choose k = 4.")

### Step 3: Fit K-Means with k = 4

In [ ]:
km_course = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE)
course_df['Cluster'] = km_course.fit_predict(course_scaled)

print("Sections per cluster:")
print(course_df['Cluster'].value_counts().sort_index())

### Step 4: PCA for Visualization

#### What Is PCA? (The Intuition)

Our courses live in **12 dimensions** — one per variable. We can't draw a 12-dimensional
scatter plot, so we need to flatten it down to 2 dimensions we *can* plot. The naive way
would be to just pick two variables (say DFW rate and enrollment) and ignore the other
eleven — but then we'd be throwing away most of what makes courses different.

**Principal Component Analysis (PCA)** is the smart way to flatten. The intuition:

> Think of the 12-dimensional cloud of courses as a 3-D object you're photographing.
> Some camera angles are informative (you see its full shape) and some are useless (you
> stare straight down the length and it looks like a dot). **PCA finds the single most
> informative angle to photograph from** — the direction in which the courses are most
> spread out — calls that **PC1**. Then it finds the best *remaining* angle at right
> angles to the first and calls that **PC2**. Plotting PC1 vs. PC2 is the "best possible
> 2-D photograph" of the 12-D cloud.

A few plain-terms takeaways:

- **Each principal component is a blend of the original variables**, not a single one.
  PC1 might roughly mean "overall course health" (mixing DFW, pass rate, GPA, repeats),
  because those move together.
- **The axes lose their literal labels.** "PC1 = 45%" means that one axis captures 45% of
  the total spread among courses — it does *not* mean "45% DFW rate." We read the picture
  for **separation** (do the clusters sit apart?), not for exact values.
- **PCA here is purely for *seeing* the result.** The clustering itself used all 13
  variables; PCA just gives us a faithful 2-D snapshot to eyeball whether the groups are
  genuinely distinct.

If the four clusters land in visibly separate regions of this plot, that's strong visual
evidence the segmentation is real and not an artifact.

In [ ]:
pca_c = PCA(n_components=2, random_state=RANDOM_STATE)
course_pca = pca_c.fit_transform(course_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(course_pca[:, 0], course_pca[:, 1],
                     c=course_df['Cluster'], cmap='Set2', alpha=0.6, s=30,
                     edgecolors='w', linewidth=0.3)
ax.set_xlabel(f'PC1 ({pca_c.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca_c.explained_variance_ratio_[1]:.1%})')
ax.set_title('Course Section Segments (PCA)')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout(); plt.show()

### Step 5: Cluster Profiling

A cluster label is meaningless until we describe *what makes it distinct*. We compute
the mean of every variable within each cluster, then visualize the profiles as heatmaps.

**One subtlety before we color anything.** A naive "red = high, green = low" heatmap
misleads, because our 12 variables are not all the same kind:

- **Outcome metrics with a clear direction.** High `DFW_RATE` is bad — but high
  `AVG_GPA` is *good*. A single color rule ("high = red") would paint a healthy
  high-GPA cluster red. So for the outcome panel we **align the direction first**:
  favorable metrics (`PASS_RATE`, `AVG_GPA`) are flipped internally
  so that **red always means concerning**, whichever way the raw metric points. The
  numbers printed in each cell are still the *raw* cluster means.
- **Descriptive context with no direction.** `ENROLLMENT`, `PCT_STEM`,
  `PREREQUISITE_COUNT`, `IS_GATEWAY`… are neither good nor bad — they just describe
  *what kind of course* lives in each cluster. Coloring them good/bad would be
  meaningless, so they get a **neutral blue scale**: darker simply means "above the
  average course," lighter means below.

Two panels, two honest color rules:

| Panel | Variables | Color means |
|:------|:----------|:------------|
| **Outcomes** | DFW, repeat, repeat delay, pass, GPA | Red = concerning, green = favorable (direction-aligned) |
| **Context** | Enrollment, size, % first-year, % STEM, prereqs, gateway, units | Darker blue = above average (descriptive only) |

In [ ]:
# Cluster profile: mean of each variable per cluster
profile_c = course_df.groupby('Cluster').mean().round(3)
profile_c['Count'] = course_df['Cluster'].value_counts().sort_index()
print("=" * 100)
print("CLUSTER PROFILES — Course Bottleneck Detection")
print("=" * 100)
print(profile_c.to_string())
print("=" * 100)

# ---------------------------------------------------------------
# Panel 1: OUTCOME metrics — direction-aligned so red = concerning
# ---------------------------------------------------------------
outcome_cols = ['DFW_RATE', 'REPEAT_RATE', 'AVG_REPEAT_DELAY',
                'PASS_RATE', 'AVG_GPA']
# For these, HIGHER is BETTER — flip them so higher always = more concerning
favorable = ['PASS_RATE', 'AVG_GPA']

outcomes_raw = profile_c[outcome_cols]                    # raw means (for the annotations)
aligned = outcomes_raw.copy()
aligned[favorable] = -aligned[favorable]                  # flip favorable metrics
outcomes_color = (aligned - aligned.mean()) / aligned.std()  # z-score for the colors

# ---------------------------------------------------------------
# Panel 2: CONTEXT variables — neutral, descriptive only
# ---------------------------------------------------------------
context_cols = ['ENROLLMENT', 'SECTION_SIZE', 'PCT_FIRST_YEAR', 'PCT_STEM',
                'PREREQUISITE_COUNT', 'IS_GATEWAY', 'UNITS']
context_raw = profile_c[context_cols]
context_color = (context_raw - context_raw.mean()) / context_raw.std()

fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Outcomes: color = direction-aligned concern, numbers = raw means
sns.heatmap(outcomes_color, annot=outcomes_raw.values, fmt='.2f',
            cmap='RdYlGn_r', center=0, linewidths=1, ax=axes[0],
            cbar_kws={'label': 'concern (direction-aligned)'})
axes[0].set_title('Outcome Metrics — red = concerning, green = favorable (numbers are raw means)')
axes[0].set_ylabel('Cluster')

# Context: neutral blues, numbers = raw means
sns.heatmap(context_color, annot=context_raw.values, fmt='.2f',
            cmap='Blues', linewidths=1, ax=axes[1],
            cbar_kws={'label': 'vs. average course'})
axes[1].set_title('Context Variables — darker = above average (descriptive, not good/bad)')
axes[1].set_ylabel('Cluster')

plt.tight_layout(); plt.show()

### Reading the Profiles

Read the two panels together, top to bottom:

1. **Find the red row in the Outcomes panel.** Because the colors are direction-aligned,
   one cluster should light up red across the board: high DFW, high repeats, long repeat
   delays, low pass rate, low GPA. That row is your
   **bottleneck group** — and you can trust the color everywhere, because red now means
   "concerning" for every column (including GPA).
2. **Then look at the same row in the Context panel to learn *who* it is.** The dark-blue
   cells describe the bottleneck cluster's identity: large enrollments, big sections,
   heavy first-year and STEM populations, mostly gateway courses. Nothing in this panel
   is "bad" by itself — plenty of large gateway courses are healthy — but it tells the
   Curriculum Committee *where* the problem lives.

The remaining rows describe the healthy segments: small upper-division courses (many
prerequisites, few first-years), high-demand large lectures with fine outcomes, and the
mid-size healthy core.

Now we turn that qualitative read into a single ranked score.

## 4. Risk Prioritization Index

Clusters tell leadership *which kinds* of courses are problematic. To tell them *which
specific sections to review first*, we build a composite **Risk Prioritization Index**
for every section.

The index combines three weighted factors:
- **DFW Rate** (weight: 0.4) — direct measure of student failure
- **Repeat Rate** (weight: 0.3) — indicates students are stuck
- **Inverse Pass Rate** (weight: 0.3) — lower pass rate = higher risk

Each factor is normalized to 0–1 before weighting, so the final index is comparable across sections.

In [ ]:
# Risk Prioritization Index
course_df['RISK_INDEX'] = (
    0.4 * (course_df['DFW_RATE'] / course_df['DFW_RATE'].max()) +
    0.3 * (course_df['REPEAT_RATE'] / course_df['REPEAT_RATE'].max()) +
    0.3 * ((1 - course_df['PASS_RATE']) / (1 - course_df['PASS_RATE']).max())
).round(3)

# Top 20 highest-risk sections
top_risk = course_df.nlargest(20, 'RISK_INDEX')[
    ['ENROLLMENT', 'DFW_RATE', 'PASS_RATE', 'REPEAT_RATE', 'IS_GATEWAY', 'RISK_INDEX', 'Cluster']
]

print("=" * 80)
print("TOP 20 HIGHEST-RISK COURSE SECTIONS")
print("=" * 80)
print(top_risk.to_string())

# Risk distribution by cluster
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
course_df.boxplot(column='RISK_INDEX', by='Cluster', ax=axes[0])
axes[0].set_title('Risk Index by Cluster')
axes[0].set_xlabel('Cluster'); axes[0].set_ylabel('Risk Index')
plt.sca(axes[0]); plt.title('Risk Index by Cluster')

# Histogram
for c in sorted(course_df['Cluster'].unique()):
    axes[1].hist(course_df[course_df['Cluster'] == c]['RISK_INDEX'],
                 alpha=0.5, bins=20, label=f'Cluster {c}')
axes[1].set_xlabel('Risk Index'); axes[1].set_ylabel('Count')
axes[1].set_title('Risk Index Distribution by Cluster')
axes[1].legend()
plt.tight_layout(); plt.show()

### Final Capstone Deliverables

In a real institutional analysis, you would deliver:

1. **Cluster summary table** — One row per cluster with mean statistics and interpretation
2. **PCA scatter plot** — Visual proof that clusters are separable
3. **Risk Prioritization Index** — Ranked list of courses needing review
4. **Executive memo** — 1-page summary for the Provost (see template below)

#### Executive Memo Template

> **To:** Provost / Curriculum Committee
>
> **From:** Institutional Research
>
> **Re:** Course Bottleneck Analysis — Risk-Prioritized Findings
>
> Using K-Means clustering on 12 course-level metrics for 1,000 sections,
> we identified four distinct course profiles. Cluster [X] contains the highest-risk
> sections (avg DFW rate = [Y]%, avg repeat rate = [Z]%). These [N] sections
> account for [P]% of total DFW enrollments despite being only [Q]% of sections.
>
> **Recommendation:** Prioritize curriculum review for the top 20 sections
> identified by the Risk Prioritization Index (attached).